# 04 · Predicción en lote con el modelo promovido

Usa `artifacts/models/actual` (model.onnx + metadata.json) con la misma regla que el backend (`OnnxPredictor`): faltantes → `fill_values`, última salida del ONNX = probabilidades, y `unknown` si la probabilidad de la clase ganadora queda bajo su umbral. Por consola: `python main.py predecir ENTRADA.csv SALIDA.csv`.

In [1]:
import os, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)

# Raíz del paquete: la carpeta que contiene config/motor.json
RAIZ = Path.cwd()
while not (RAIZ / "config" / "motor.json").is_file() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
SAL = Path(os.environ.get("CLASIFICADOR_SALIDAS", RAIZ / "salidas"))
ENT = RAIZ / "datos_entrada" / "processed"
TY = RAIZ / "resultados_TY" / "run_004"
if not (TY / "02_particion" / "particion.parquet").is_file():
    TY = RAIZ / "resultados_TY" / "run_003"   # respaldo: corrida anterior incluida en el paquete
print("raíz:", RAIZ, "| salidas:", SAL, "| TY:", TY.name)

raíz: /home/claude/build/Modelo | salidas: /home/claude/build/Modelo/salidas | TY: run_004


In [2]:
import sys; sys.path.insert(0, str(RAIZ))
import main
MOD = RAIZ / "artifacts" / "models" / "actual"
meta = json.loads((MOD / "metadata.json").read_text(encoding="utf-8"))
print(meta["model_version"]); print("features:", meta["features"]); print("clases:", meta["classes"], "| umbrales:", meta["thresholds"])
print(json.dumps(meta["training"], indent=1, ensure_ascii=False))

motor-rf-E_servible_binario-spherex-2026-11-16T0256
features: ['cw_W1_W2', 'cw_mu', 'Hrpm_W2', 'spx_c21', 'spx_c32', 'spx_c43', 'spx_c54', 'spx_c65', 'spx_c64', 'spx_ch4']
clases: ['star', 'cold_brown_dwarf'] | umbrales: {'cold_brown_dwarf': 0.5}
{
 "experimento": "E_servible_binario",
 "modelo": "rf",
 "potencia_peso_clase": 0.25,
 "poblacion": "spherex",
 "n_dev": 697,
 "n_test": 151,
 "cv": {
  "balanced_accuracy": 0.9457,
  "macro_f1": 0.9455,
  "ap_positivas": 0.983,
  "precision_positivas": 0.9349,
  "recall_positivas": 0.9381,
  "recall_star": 0.9532,
  "recall_cold_brown_dwarf": 0.9381
 },
 "test": {
  "balanced_accuracy": 0.9583,
  "macro_f1": 0.9583,
  "ap_positivas": 0.9924,
  "precision_positivas": 0.9492,
  "recall_positivas": 0.9492,
  "recall_star": 0.9674,
  "recall_cold_brown_dwarf": 0.9492
 },
 "particion": "espacial de TY_PIPELINE (celdas de cielo; test = 20 % de celdas)",
 "etiquetas": "star = estrella, M o L; cold_brown_dwarf = T o Y (UltracoolSheet)"
}


## Lote de ejemplo: el test externo del experimento exportado

Son objetos que el modelo nunca vio (celdas de cielo de test).

In [3]:
tab = pd.read_parquet(SAL / "4_preprocesamiento" / "tabla_modelo.parquet")
lote = tab[(tab.role == "test") & tab.in_spherex][["oid", "label"] + meta["features"]].rename(columns={"label": "label_real"})
import tempfile; TMP = Path(tempfile.gettempdir())
entrada, salida = TMP / "lote_entrada.csv", TMP / "lote_predicciones.csv"
lote.to_csv(entrada, index=False)
class A: pass
a = A(); a.entrada, a.salida, a.modelo = str(entrada), str(salida), str(MOD)
main.cmd_predecir(a)
pred = pd.read_csv(salida)
mapa = {"star": "star", "M": "star", "L": "star", "T": "cold_brown_dwarf", "Y": "cold_brown_dwarf"}
pred["real"] = pred["label_real"].map(mapa)
print(pd.crosstab(pred["real"], pred["label"], rownames=["real"], colnames=["predicha"], margins=True))

151 filas -> /tmp/lote_predicciones.csv | {'star': 92, 'cold_brown_dwarf': 59}
predicha          cold_brown_dwarf  star  All
real                                         
cold_brown_dwarf                56     3   59
star                             3    89   92
All                             59    92  151


In [4]:
pred.sort_values("p_cold_brown_dwarf", ascending=False).head(10)[["oid", "real", "p_cold_brown_dwarf"] + meta["features"][:4]]

,oid,real,p_cold_brown_dwarf,cw_W1_W2,cw_mu,Hrpm_W2,spx_c21
9,235,cold_brown_dwarf,0.999994,2.277,0.515873,18.515712,NaN
107,3581,cold_brown_dwarf,0.999994,2.085,0.374467,18.121066,NaN
105,3417,cold_brown_dwarf,0.999994,3.591,1.013918,19.346015,NaN
96,3233,cold_brown_dwarf,0.999994,4.806,1.109725,19.834078,NaN
108,3582,cold_brown_dwarf,0.999994,2.788,0.406339,18.120441,NaN
41,1323,cold_brown_dwarf,0.999994,2.532,0.145703,15.962342,NaN
48,1824,cold_brown_dwarf,0.999994,2.829,1.522025,21.229109,NaN
76,2630,cold_brown_dwarf,0.999597,3.230,1.224946,21.650585,NaN
47,1783,cold_brown_dwarf,0.999597,3.242,0.583213,20.096137,NaN
15,466,cold_brown_dwarf,0.999597,3.151,0.647488,20.524158,NaN


## Un objeto sin datos

Si faltan todas las variables, el modelo recibe solo los valores de relleno (medianas del entrenamiento). Con este modelo eso da cold_brown_dwarf, porque el subconjunto SPHEREx tiene 42 % de enanas frías. Por eso el backend debe marcar como `insufficient_data` las fuentes sin `cw_W1_W2` en lugar de usar la predicción.

In [5]:
meta2, sess = main._cargar(MOD)
print(dict(zip(meta2["classes"], main._probabilidades(meta2, sess, [{}])[0].round(3))))

{'star': np.float64(0.216), 'cold_brown_dwarf': np.float64(0.784)}
